In [ ]:
import os
import json
import pandas as pd

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
KEY= os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [ ]:
model = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    provider= "featherless-ai",
    max_new_tokens=1500,
    stop=["\n[1]", "\n\n["],
    repetition_penalty=1.03,
    huggingfacehub_api_token=KEY
)

In [ ]:
llm = ChatHuggingFace(llm=model, temperature=0.3)

In [ ]:
llm

In [ ]:
from langchain_core.prompts import PromptTemplate
import PyPDF2

In [ ]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [ ]:
TEMPLATE = """
Text: {text}
You are an expert MCQ maker. Given the above text, create EXACTLY {number} multiple choice questions for {subject} students in {tone} tone.
Respond with ONLY the JSON object below, filled in — no explanations, no citations, no text before or after the JSON.
Make sure the questions are not repeated and check all the questions for grammar and spelling mistakes and to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \

### RESPONSE_JSON
{response_json}
"""

In [ ]:
quiz_generator_template = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template= TEMPLATE
)

In [ ]:
quiz_chain= quiz_generator_template|llm | StrOutputParser()

In [ ]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [ ]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [ ]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [ ]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [ ]:
file_path= r"C:\Users\PARNA JAIN\mcqgen\data.txt"

In [ ]:
with open(file_path, "r") as file:
    text = file.read()

In [ ]:
json.dumps(RESPONSE_JSON)

In [ ]:
response= generate_evaluate_chain.invoke(
    {
        "text": text,
        "number": 5,
        "subject": "Artificial Intelligence",
        "tone": "simple",
        "response_json": json.dumps(RESPONSE_JSON),
    }
)

In [ ]:
response

In [ ]:
quiz= response.get("quiz")

In [ ]:
print(len(quiz))
print(quiz)

In [ ]:
decoder = json.JSONDecoder()
start = quiz.index('{')
quiz_json, _ = decoder.raw_decode(quiz, start)

In [ ]:
quiz_json